# Retrain v4: validate that v0.34 cosine fix did not change training signal

**Hypothesis under test (2026-04-28).** v2 and v3 were trained on pre-v0.34 vstash code where `vec_chunks` used L2 distance under a `cosine` label. The v0.34 cosine metric fix (#271/#272/#286) recalibrated `distance_cutoff` and `relevance_tier` thresholds.

For BGE-small unit-normalized embeddings, cosine and L2 are monotonically related (L2^2 = 2 * (1 - cos)). Ranking by L2 vs cosine gives the same top-K order, and the `1.15x` L2 cutoff equals the `1.3225x` cosine cutoff (since 1.15^2 = 1.3225). **In theory the training triples should be invariant to the fix.**

This notebook trains a v4 model using the **same recipe as v3 (H-R9 winning config: temperature=0.5, total_triples=60000)** but on the **v0.35 develop branch (post-cosine-fix)**, then runs the BEIR benchmark.

- If v4 macro NDCG@10 is within the bootstrap CI of v3 (i.e. delta within ~0.005 absolute), the training-side invariance holds and the paper can use v3 numbers with confidence.
- If v4 macro NDCG@10 differs from v3 substantially, a training-side effect exists in addition to the eval-side cosine fix, and the paper should report v4 as the post-v0.34 reference model.

Wall time on Colab T4: ~12 min training + ~15 min BEIR eval = ~30 min total.

In [ ]:
# Cell 1: Setup -- clone develop (v0.35.0+, post-cosine-fix).
!pip install -q 'sentence-transformers>=3' torch 'accelerate>=1.1.0'
!rm -rf /content/vstash
!git clone --branch develop https://github.com/stffns/vstash.git /content/vstash
%cd /content/vstash
!pip install -q -e .
# Sanity check: confirm we are on v0.35+ (cosine fix present).
!python -c "import vstash; print('vstash', vstash.__version__)"
!grep -E 'distance_metric.*cosine|SCHEMA_VERSION' vstash/store.py | head -3

In [ ]:
# Cell 2: Download BEIR + ingest into per-dataset stores using BGE-small base.
import os
import sys

os.chdir("/content/vstash")
sys.path.insert(0, "/content/vstash")
os.makedirs("experiments/data", exist_ok=True)

from experiments.beir_benchmark import download_beir, load_beir  # noqa: E402
from sentence_transformers import SentenceTransformer  # noqa: E402
from vstash.store import VstashStore  # noqa: E402

BASE_MODEL = "BAAI/bge-small-en-v1.5"
# v3 was trained on these 3 datasets (per retrain_t1_5_hr9_balance_ablation.ipynb).
# v4 keeps the same training set for apples-to-apples comparison.
DATASETS = ["scifact", "nfcorpus", "fiqa"]

encoder = SentenceTransformer(BASE_MODEL, device="cuda")
stores = {}
per_dataset = {}

for name in DATASETS:
    cache = download_beir(name)
    corpus, queries, qrels = load_beir(cache)
    print(f"[{name}] corpus={len(corpus)} queries={len(queries)} qrels={len(qrels)}")
    db_path = f"/content/store_{name}.db"
    if os.path.exists(db_path):
        os.remove(db_path)
    store = VstashStore(db_path, embedding_dim=384)
    # Bulk ingest with GPU embeddings.
    doc_ids = list(corpus.keys())
    BATCH = 256
    for i in range(0, len(doc_ids), BATCH):
        batch_ids = doc_ids[i : i + BATCH]
        texts = [
            (corpus[d].get("title", "") + "\n" + corpus[d].get("text", "")).strip()
            for d in batch_ids
        ]
        embs = encoder.encode(texts, normalize_embeddings=True, show_progress_bar=False)
        store.add_documents_batch(
            [
                {
                    "path": f"{name}://{doc_id}",
                    "title": corpus[doc_id].get("title", ""),
                    "chunks": [text],
                    "embeddings": [emb.tolist()],
                    "source_type": "text",
                }
                for doc_id, text, emb in zip(batch_ids, texts, embs)
            ]
        )
        if (i // BATCH) % 20 == 0:
            print(f"  {name}: {i + len(batch_ids)}/{len(doc_ids)}")
    stores[name] = store
    per_dataset[name] = {"queries": queries, "qrels": qrels}
    print(f"  {name}: ingested {store.stats().chunks} chunks")

In [ ]:
# Cell 3: Build per-dataset eval queries from real qrels (T1.5 v5 recipe).
from vstash.retrain import qrels_to_eval_queries

eval_queries_by_dataset = {}
for name, bundle in per_dataset.items():
    eqs = qrels_to_eval_queries(
        queries=bundle["queries"],
        qrels=bundle["qrels"],
        path_for_doc_id=lambda doc_id, d=name: f"{d}://{doc_id}",
    )
    eval_queries_by_dataset[name] = eqs
    print(f"[{name}] eval_queries (real qrels): {len(eqs)}")

In [ ]:
# Cell 4: Train v4 with the v3 winning config (arm_vol from H-R9 ablation):
# temperature=0.5, total_triples=60000.
import time
import shutil
from vstash.retrain import retrain_multi

EVAL_NOISE = max(max(s.stats().chunks for s in stores.values()), 10000)
SEED = 42
OUTPUT_PATH = "/content/retrained_v4_post_v034"

# Clean any prior partial output
for suffix in ("", ".candidate", ".old"):
    p = OUTPUT_PATH + suffix
    if os.path.exists(p):
        shutil.rmtree(p)

t0 = time.time()
result_v4 = retrain_multi(
    stores=stores,
    base_model=BASE_MODEL,
    output_path=OUTPUT_PATH,
    sampling="temperature",
    temperature=0.5,
    total_triples=60000,
    epochs=2,
    lr=3e-6,
    batch_size=64,
    eval_noise_size=EVAL_NOISE,
    bulk_mine=True,
    bulk_mine_device="cuda",
    seed=SEED,
    eval_queries_by_dataset=eval_queries_by_dataset,
    min_gain=-1.0,  # save even if regressing; we want the artifact for comparison
)
elapsed = time.time() - t0
print(f"Training wall time: {elapsed / 60:.1f} min")
print(f"Output: {OUTPUT_PATH}")
print(f"Result: {result_v4}")

In [ ]:
# Cell 5: Run BEIR benchmark on v4 (5 datasets, ST-CUDA backend).
# Uses the patched beir_benchmark.py that emits per-query NDCG sidecar.
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Register the v4 path with vstash.embed so the resolver can find it.
# Then BEIR benchmark via the existing CLI.
!cd /content/vstash && PYTHONPATH=/content/vstash python -m experiments.beir_benchmark \
    --no-chroma \
    --model /content/retrained_v4_post_v034 \
    --device cuda 2>&1 | grep -vE '^encoder\\.|^embeddings\\.|^Loading weights:|^FutureWarning|gamma,$|beta,$' | tail -50

In [ ]:
# Cell 6: Compare v4 vs v3 on macro and per dataset.
# v3 numbers from the local Mac run (2026-04-28, v0.35 ST-CPU):
v3_v035 = {
    "scifact": 0.7705,
    "nfcorpus": 0.3755,
    "fiqa": 0.4648,
    "scidocs": 0.1954,
    "arguana": 0.4318,
}
v3_macro = sum(v3_v035.values()) / len(v3_v035)

import json

with open("/content/vstash/experiments/results/beir_benchmark.json") as f:
    v4_data = json.load(f)
v4 = {r["dataset"]: r["vstash"]["ndcg_10"] for r in v4_data["results"]}
v4_macro = sum(v4.values()) / len(v4)

print(f"{'dataset':<12} {'v3 (CPU)':>10} {'v4 (CUDA)':>10} {'delta':>10}")
print("-" * 48)
for d in sorted(v4):
    delta = v4[d] - v3_v035[d]
    print(f"{d:<12} {v3_v035[d]:>10.4f} {v4[d]:>10.4f} {delta:>+10.4f}")
print("-" * 48)
print(f"{'macro':<12} {v3_macro:>10.4f} {v4_macro:>10.4f} {v4_macro - v3_macro:>+10.4f}")

# Backend caveat: v3 numbers above were measured with ST-CPU on Mac.
# v4 is measured with ST-CUDA on Colab. There can be a small numerical
# difference (~0.001-0.003 NDCG@10). Treat absolute values within
# +/-0.005 macro as 'within noise'.

In [ ]:
# Cell 7: Download v4 + aggregate result for review on Mac.
# Per-query sidecar is optional (only emitted if develop has the
# 2026-04-28 per-query patch; otherwise just aggregates).
from google.colab import files  # noqa: F401 -- only present in Colab
import os

# Bundle the trained model so we can re-run paired bootstrap on Mac
# against the existing v3 / base / lme-v1 sidecars.
!cd /content && tar czf retrained_v4_post_v034.tar.gz retrained_v4_post_v034
files.download("/content/retrained_v4_post_v034.tar.gz")

# Aggregate BEIR result.
files.download("/content/vstash/experiments/results/beir_benchmark.json")

# Optional: per-query sidecar (only present if develop has the
# 2026-04-28 per-query patch). Safe-skip if missing.
results_dir = "/content/vstash/experiments/results"
for f in sorted(os.listdir(results_dir)):
    if f.startswith("beir_perquery_"):
        files.download(os.path.join(results_dir, f))